In [19]:
# ================================================
#   INSTALACIÓN DE DEPENDENCIAS
# ================================================
!pip install segment-anything opencv-python pytesseract transformers easyocr tqdm pandas --quiet
!pip install git+https://github.com/facebookresearch/segment-anything.git --quiet
!pip install -q datasets evaluate accelerate python-doctr[torch] jiwer

# Descargar checkpoint de SAM
!wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -O sam_vit_b.pth

# Montar Google Drive y extraer ZIP
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os
zip_path = "/content/drive/MyDrive/Colab Notebooks/MNA/Proyecto Integrador/Copia de 20260121 imagenes Cashcollection1.zip"
extract_dir = "/content/drive/MyDrive/Colab Notebooks/MNA/Proyecto Integrador/dataset_bimbonet"
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)

IMAGE_DIR = os.path.join(extract_dir, "20260121 imagenes Cashcollection")
print("Carpeta de imágenes:", IMAGE_DIR)
print("Ejemplo de archivos:", os.listdir(IMAGE_DIR)[:5])

  Preparing metadata (setup.py) ... done
--2026-03-02 08:58:32--  https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 13.249.182.62, 13.249.182.39, 13.249.182.81, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|13.249.182.62|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 375042383 (358M) [binary/octet-stream]
Saving to: ‘sam_vit_b.pth’

sam_vit_b.pth       100%[===================>] 357.67M   263MB/s    in 1.4s    

2026-03-02 08:58:33 (263 MB/s) - ‘sam_vit_b.pth’ saved [375042383/375042383]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Carpeta de imágenes: /content/drive/MyDrive/Colab Notebooks/MNA/Proyecto Integrador/dataset_bimbonet/20260121 imagenes Cashcollection
Ejemplo de archivos: ['base64decoded - 2026-01-21T153732.177.png', 'base64decoded - 2026-01-21T153741.826.png', 'bas

In [20]:
# ================================================
#   IMPORTS Y DEVICE
# ================================================
import cv2, torch, numpy as np, pytesseract, re, pandas as pd
from tqdm import tqdm
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
import easyocr
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, TrOCRProcessor, VisionEncoderDecoderModel, Seq2SeqTrainer, Seq2SeqTrainingArguments

device = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", device)


DEVICE: cuda


In [21]:
# ================================================
#   CARGA DE MODELOS
# ================================================
# SAM
sam = sam_model_registry["vit_b"](checkpoint="sam_vit_b.pth").to(device)
mask_generator = SamAutomaticMaskGenerator(
    model=sam, points_per_side=16,
    pred_iou_thresh=0.6,
    stability_score_thresh=0.5,
    min_mask_region_area=50
)

# DINO
processor_dino = AutoProcessor.from_pretrained("IDEA-Research/grounding-dino-base")
dino = AutoModelForZeroShotObjectDetection.from_pretrained("IDEA-Research/grounding-dino-base").to(device)

# EasyOCR
reader = easyocr.Reader(['es','en'], gpu=(device=="cuda"))

# TrOCR
processor_trocr = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")
model_trocr = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-printed").to(device)

Loading weights:   0%|          | 0/1206 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-printed
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [22]:
# ================================================
#   FUNCIONES DE EXTRACCIÓN
# ================================================
def extract_col_strict(text):
    text = text.upper()
    pattern = r"C[O0]L\s*([123])[\.\,\s]*(\d{5})\s+(\d{5})"
    matches = re.findall(pattern, text)
    return [f"COL {m[0]}. {m[1]} {m[2]}" for m in matches]

def sam_extract(img):
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_small = cv2.resize(img_rgb, (1024, 1024))
    masks = mask_generator.generate(img_small)
    found = []
    for m in masks:
        mask = m["segmentation"].astype(np.uint8)*255
        x, y, w, h = cv2.boundingRect(mask)
        crop = img_small[y:y+h, x:x+w]
        if crop.size==0: continue
        ocr = pytesseract.image_to_string(crop)
        cleaned = extract_col_strict(ocr)
        if cleaned: found.extend(cleaned)
    return sorted(list(set(found)))

def trocr_extract(crop_img):
    if crop_img is None or crop_img.size == 0: return []
    pil_img = Image.fromarray(cv2.cvtColor(crop_img, cv2.COLOR_BGR2RGB))
    pixel_values = processor_trocr(pil_img, return_tensors="pt").pixel_values.to(device)
    with torch.no_grad():
        generated_ids = model_trocr.generate(pixel_values)
    text = processor_trocr.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return extract_col_strict(text)

def dino_fallback(img):
    if img is None or img.size==0: return []
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    inputs = processor_dino(images=img_rgb, text="COL number line text numbers", return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = dino(**inputs)
    results_all = processor_dino.post_process_grounded_object_detection(outputs, inputs.input_ids)
    results = results_all[0]
    threshold=0.22
    found=[]
    h,w=img.shape[:2]
    for score, box in zip(results["scores"], results["boxes"]):
        if score<threshold: continue
        x1,y1,x2,y2 = map(int, box.tolist())
        x1,y1=max(0,x1), max(0,y1)
        x2,y2=min(w,x2), min(h,y2)
        if x2<=x1 or y2<=y1: continue
        crop = img[y1:y2, x1:x2]
        try:
            ocr_list = reader.readtext(crop, detail=0)
            found.extend(extract_col_strict(" ".join(ocr_list)))
        except: pass
        try:
            found.extend(trocr_extract(crop))
        except: pass
    return sorted(list(set(found)))

def extract_col_hybrid(img_path):
    img = cv2.imread(img_path)
    sam_result = sam_extract(img)
    if sam_result: return sam_result
    dino_result = dino_fallback(img)
    if dino_result: return dino_result
    return trocr_extract(img)

In [24]:
# ================================================
#   SUBSET PARA PROCESAMIENTO RÁPIDO
# ================================================
all_images = sorted([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith((".jpg",".jpeg",".png",".jfif"))])
subset_images = all_images[:10]  # 🔹 usar 50 imágenes
rows=[]
for img_name in tqdm(subset_images, desc="Procesando imágenes"):
    path = os.path.join(IMAGE_DIR,img_name)
    detected = extract_col_hybrid(path)
    col1 = next((x for x in detected if x.startswith("COL 1")),"")
    col2 = next((x for x in detected if x.startswith("COL 2")),"")
    col3 = next((x for x in detected if x.startswith("COL 3")),"")
    rows.append({"imagen": img_name,"col1": col1,"col2": col2,"col3": col3})
df_results = pd.DataFrame(rows)
csv_path = "/content/drive/MyDrive/Colab Notebooks/MNA/Proyecto Integrador/resultados_extraccion.csv"
df_results.to_csv(csv_path,index=False,encoding="utf-8")
print("CSV guardado en:", csv_path)

Procesando imágenes: 100%|██████████| 10/10 [05:25<00:00, 32.57s/it]

CSV guardado en: /content/drive/MyDrive/Colab Notebooks/MNA/Proyecto Integrador/resultados_extraccion.csv


In [28]:
# ================================================
#   CREAR DATASET PARA FINE-TUNING TrOCR
# ================================================
# Directorio para guardar los recortes
dataset_dir = "dataset_ocr/images"
os.makedirs(dataset_dir, exist_ok=True)

# Diccionario de etiquetas
labels_doctr = {}
df_train_data = []

# Ajustamos a 50 imágenes para prueba rápida
LIMIT_TRAIN = 10
subset_images = sorted([
    f for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".jfif"))
])[:LIMIT_TRAIN]

for idx, img_name in enumerate(tqdm(subset_images, desc="Generando dataset TrOCR")):
    path = os.path.join(IMAGE_DIR, img_name)
    img_cv = cv2.imread(path)
    if img_cv is None:
        continue
    img_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)

    # --- 1) Intentamos recortes SAM ---
    masks = mask_generator.generate(cv2.resize(img_rgb, (1024, 1024)))
    recortes_sam = []

    for m_idx, m in enumerate(masks):
        mask = m["segmentation"].astype(np.uint8) * 255
        x, y, w, h = cv2.boundingRect(mask)
        crop = img_rgb[y:y+h, x:x+w]
        if crop.shape[0] < 5 or crop.shape[1] < 5:
            continue
        # OCR rápido con EasyOCR para etiqueta
        try:
            texto_detectado = " ".join(reader.readtext(crop, detail=0)).strip()
        except:
            texto_detectado = ""
        if texto_detectado:
            filename = f"recorte_{idx}_sam_{m_idx}.jpg"
            save_path = os.path.join(dataset_dir, filename)
            Image.fromarray(crop).save(save_path)
            df_train_data.append({"image_path": save_path, "text": texto_detectado})
            labels_doctr[filename] = texto_detectado
            recortes_sam.append(True)

    # --- 2) Fallback DINO + EasyOCR solo si SAM no detectó nada ---
    if not recortes_sam:
        inputs = processor(images=img_rgb, text="printed text numbers.", return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = dino(**inputs)
        results = processor.post_process_grounded_object_detection(outputs, inputs.input_ids)[0]

        for box_idx, box in enumerate(results["boxes"]):
            x1, y1, x2, y2 = map(int, box.tolist())
            y1, y2 = max(0, y1-2), min(img_rgb.shape[0], y2+2)
            x1, x2 = max(0, x1-2), min(img_rgb.shape[1], x2+2)
            crop = img_rgb[y1:y2, x1:x2]
            if crop.shape[0] < 5 or crop.shape[1] < 5:
                continue
            try:
                texto_detectado = " ".join(reader.readtext(crop, detail=0)).strip()
            except:
                texto_detectado = ""
            if texto_detectado:
                filename = f"recorte_{idx}_dino_{box_idx}.jpg"
                save_path = os.path.join(dataset_dir, filename)
                Image.fromarray(crop).save(save_path)
                df_train_data.append({"image_path": save_path, "text": texto_detectado})
                labels_doctr[filename] = texto_detectado

# Guardar dataset
df_train = pd.DataFrame(df_train_data)
with open("dataset_ocr/labels.json", "w") as f:
    json.dump(labels_doctr, f)

print(f"✅ Dataset TrOCR listo con {len(df_train)} recortes (SAM principal, DINO fallback)")

Generando dataset TrOCR: 100%|██████████| 10/10 [01:40<00:00, 10.01s/it]

✅ Dataset TrOCR listo con 234 recortes (SAM principal, DINO fallback)


In [32]:
# ================================================
# Fine-Tuning TrOCR con dataset generado
# ================================================
from transformers import VisionEncoderDecoderModel, Seq2SeqTrainer, Seq2SeqTrainingArguments
import torch

# Modelo base
model_trocr = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-printed")
model_trocr.config.decoder_start_token_id = processor_trocr.tokenizer.cls_token_id
model_trocr.config.pad_token_id = processor_trocr.tokenizer.pad_token_id
model_trocr.to(device)

# Dataset
train_dataset = TrOCRDataset(df_train, processor_trocr)

# Configuración de entrenamiento
training_args = Seq2SeqTrainingArguments(
    output_dir="./trocr_bimbo_finetuned",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    num_train_epochs=2,
    predict_with_generate=True,
    save_strategy="no",
    logging_steps=5,
    fp16=torch.cuda.is_available()
)

# Seq2SeqTrainer sin 'tokenizer'
trainer = Seq2SeqTrainer(
    model=model_trocr,
    args=training_args,
    train_dataset=train_dataset
)

# Entrenar
trainer.train()
print("✅ Fine-Tuning de TrOCR completado")

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-printed
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
5,10.639207
10,8.690076
15,6.815776
20,6.829113
25,6.602357
30,6.269413
35,4.103715
40,5.912858
45,5.802792
50,5.962694


✅ Fine-Tuning de TrOCR completado


In [34]:
# ================================================
#  Actualizar función trocr_extract con modelo fine-tuneado
# ================================================
def trocr_extract(img):
    """
    Extrae COLs usando el modelo TrOCR fine-tuneado.
    """
    if img is None or img.size == 0:
        return []

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img_rgb)
    pixel_values = processor_trocr(pil_img, return_tensors="pt").pixel_values.to(device)

    with torch.no_grad():
        generated_ids = model_trocr.generate(pixel_values)

    text_pred = processor_trocr.batch_decode(generated_ids, skip_special_tokens=True)[0].upper()
    # Regex para detectar COLs
    pattern = r"C[O0]L\s*([123])[\.\,\s]*(\d{5})\s+(\d{5})"
    matches = re.findall(pattern, text_pred)
    return [f"COL {m[0]}. {m[1]} {m[2]}" for m in matches]

# ================================================
#  Ejecutar pipeline híbrido en 50 imágenes
# ================================================
all_images = sorted([
    f for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".jfif"))
])

subset_images = all_images[:20]  # Subset de 50 imágenes
rows = []

for img_name in tqdm(subset_images, desc="Procesando imágenes con TrOCR fine-tuneado"):
    path = os.path.join(IMAGE_DIR, img_name)
    detected = extract_col_hybrid(path)  # SAM principal, DINO fallback, TrOCR fine-tuneado
    col1 = next((x for x in detected if x.startswith("COL 1")), "")
    col2 = next((x for x in detected if x.startswith("COL 2")), "")
    col3 = next((x for x in detected if x.startswith("COL 3")), "")
    rows.append({"imagen": img_name, "col1": col1, "col2": col2, "col3": col3})

df_results = pd.DataFrame(rows)

# Guardar CSV
csv_path = "/content/drive/MyDrive/Colab Notebooks/MNA/Proyecto Integrador/resultados_extraccion_finetuned.csv"
df_results.to_csv(csv_path, index=False, encoding="utf-8")
print("CSV guardado en:", csv_path)

# Mostrar primeras filas
df_results.head()

Procesando imágenes con TrOCR fine-tuneado: 100%|██████████| 20/20 [09:04<00:00, 27.24s/it]

CSV guardado en: /content/drive/MyDrive/Colab Notebooks/MNA/Proyecto Integrador/resultados_extraccion_finetuned.csv


,imagen,col1,col2,col3
0,1.jpeg,COL 1. 15355 50899,COL 2. 01146 94344,COL 3. 01816 44947
1,10.jpeg,,,
2,100.jpg,COL 1. 24421 29378,COL 2. 30669 56415,COL 3. 69800 86140
3,101.png,COL 1. 96476 05643,COL 2. 00069 36425,COL 3. 60880 89364
4,102.png,COL 1. 30556 59450,COL 2. 51144 64548,COL 3. 91826 13566


 VISUALIZACIÓN DE MÁSCARAS
================================================

In [35]:
import matplotlib.pyplot as plt

def visualize_sam_masks(img_path):
    img = cv2.imread(img_path)
    if img is None: return
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_small = cv2.resize(img_rgb, (1024, 1024))
    masks = mask_generator.generate(img_small)
    overlay = img_small.copy()
    for m in masks:
        mask = m["segmentation"]
        color = np.random.randint(0, 255, (3,), dtype=np.uint8)
        overlay[mask] = overlay[mask] * 0.4 + color * 0.6
    plt.figure(figsize=(10, 10))
    plt.imshow(overlay)
    plt.title("Máscaras SAM superpuestas")
    plt.axis("off")
    plt.show()

def visualize_dino(img_path):
    img = cv2.imread(img_path)
    if img is None: return
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    inputs = processor(images=img_rgb, text="printed text", return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = dino(**inputs)
    results = processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids,
        threshold=0.25, text_threshold=0.25,
        target_sizes=[img.shape[:2]]
    )[0]
    plt.figure(figsize=(10, 10))
    plt.imshow(img_rgb)
    for box in results["boxes"]:
        x1, y1, x2, y2 = map(int, box.tolist())
        plt.gca().add_patch(
            plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="red", linewidth=2)
        )
    plt.title("Detecciones de DINO")
    plt.axis("off")
    plt.show()

# Probar en primeras N imágenes
N_TEST_IMAGES = 5
test_images = [os.path.join(IMAGE_DIR, f) for f in all_images[:N_TEST_IMAGES]]

for img_path in test_images:
    print(f"Procesando: {img_path}")
    visualize_sam_masks(img_path)
    visualize_dino(img_path)
    print("-"*60)

Output hidden; open in https://colab.research.google.com to view.

In [37]:
import nbformat

input_nb = "/content/drive/MyDrive/Colab Notebooks/Copia de Copia de Avance 5_Equipo 31.ipynb"
output_nb = "/content/drive/MyDrive/Colab Notebooks/MNA/Proyecto Integrador/Copia de Copia de Avance 5_Equipo 31_clean.ipynb"

nb = nbformat.read(input_nb, as_version=4)

if "widgets" in nb.get("metadata", {}):
    del nb["metadata"]["widgets"]

for cell in nb["cells"]:
    if "metadata" in cell and "widgets" in cell["metadata"]:
        del cell["metadata"]["widgets"]

with open(output_nb, "w", encoding="utf-8") as f:
    nbformat.write(nb, f)

print("Notebook limpio guardado en:", output_nb)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Colab Notebooks/MNA/Proyecto Integrador/Copia de Copia de Avance 5_Equipo 31.ipynb'